# PMM Dynamic Multi-Pair Sweep

**Automated optimization across all available trading pairs**

This notebook:
1. Discovers all available pairs for a connector + quote asset from MongoDB
2. For each pair with sufficient data:
   - Runs 3000 Optuna trials (walk-forward, stress OFF)
   - Stress-tests the top 50 candidates
   - Evaluates the best stress-validated candidate
3. Exports YAML configs and reports for profitable pairs
4. Displays a summary comparison across all pairs

**Configuration:** Edit the variables in the first code cell, then Run All.

In [1]:
import sys, os, subprocess, time, logging
from datetime import datetime

PMM_DIR = "/quants-lab/research_notebooks/market_lab/pmm_dynamic"
if PMM_DIR not in sys.path:
    sys.path.insert(0, PMM_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PMM_DIR, "--quiet"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")

pmm_lab 0.1.0 | NumPy 2.2.6 | Optuna 4.7.0
MONGO_URI      : SET
OPTUNA_STORAGE : SET


## 1. Configuration

Edit these variables to control the sweep. Then **Run All** cells below.

In [2]:
# ==============================================================
# SWEEP CONFIGURATION — edit these, then Run All
# ==============================================================

CONNECTOR = "nonkyc"           # Exchange connector to sweep
QUOTE_ASSET = "USDT"           # Quote asset filter (pairs ending in -USDT)
N_TRIALS = 3000                # Optuna trials per pair
TOP_N = 50                     # Top candidates to stress test
MIN_ROBUST_SCORE = 0.0         # Minimum robust score to export (0 = breakeven)

# --- Parallelism ---
N_WORKERS = 4                  # Number of parallel PROCESSES (not threads)
WORKER_MODEL = "processes"     # "processes" (recommended) or "threads"

# --- Feature computation ---
PHASE1_CONTROLLER_COMPAT = False  # Fast vectorized mode for search
PHASE2_CONTROLLER_COMPAT = True   # Controller-equivalent mode for final validation

# Preferred interval per connector
CONNECTOR_INTERVALS = {
    "nonkyc": "5m",
    "mexc": "5m",
}

# Minimum data requirement (days)
MIN_DATA_DAYS = 28

# ==============================================================

INTERVAL = CONNECTOR_INTERVALS.get(CONNECTOR, "5m")

from pmm_lab.config.defaults import INTERVAL_SECONDS
BAR_INTERVAL_SECONDS = INTERVAL_SECONDS[INTERVAL]

print(f"Connector        : {CONNECTOR}")
print(f"Quote asset      : {QUOTE_ASSET}")
print(f"Interval         : {INTERVAL} ({BAR_INTERVAL_SECONDS}s/bar)")
print(f"Trials/pair      : {N_TRIALS}")
print(f"Top-N stress     : {TOP_N}")
print(f"Min score        : {MIN_ROBUST_SCORE}")
print(f"Min data days    : {MIN_DATA_DAYS}")
print(f"Worker model     : {WORKER_MODEL}")
print(f"N workers        : {N_WORKERS}")
print(f"Phase 1 compat   : {PHASE1_CONTROLLER_COMPAT}")
print(f"Phase 2 compat   : {PHASE2_CONTROLLER_COMPAT}")

Connector        : nonkyc
Quote asset      : USDT
Interval         : 5m (300s/bar)
Trials/pair      : 3000
Top-N stress     : 50
Min score        : 0.0
Min data days    : 28
Worker model     : processes
N workers        : 4
Phase 1 compat   : False
Phase 2 compat   : True


In [3]:
# ==============================================================
# PREFLIGHT CHECKS — validates environment before optimization
# ==============================================================

from pmm_lab.optuna.preflight import print_environment, run_preflight

print_environment()
print()

preflight = run_preflight(
    n_workers=N_WORKERS,
    storage_url=OPTUNA_STORAGE if OPTUNA_STORAGE else None,
    worker_model=WORKER_MODEL,
    strict=True,  # Fail early if config is wrong
)

Python     : 3.12.13
NumPy      : 2.2.6
Pandas     : 3.0.1
Optuna     : 4.7.0
pmm_lab    : 0.1.0
Storage    : PostgreSQL (SET)
CPU cores  : 32
OMP_NUM_THREADS          : 1
OPENBLAS_NUM_THREADS     : 1
MKL_NUM_THREADS          : 1
NUMEXPR_NUM_THREADS      : 1

Preflight: ALL CHECKS PASSED


## 2. Discover Available Pairs

In [4]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles

loader = MongoCandleLoader()
all_combos = loader.list_combos(connector=CONNECTOR, quote_asset=QUOTE_ASSET)

# Filter to our selected interval and minimum data
candidates = []
for combo in all_combos:
    if combo["interval"] != INTERVAL:
        continue
    data_days = (combo["last_ts"] - combo["first_ts"]) / 86400
    if data_days < MIN_DATA_DAYS:
        print(f"  SKIP {combo['trading_pair']:15s}  {combo['count']:>8,} candles  {data_days:5.1f} days (< {MIN_DATA_DAYS}d)")
        continue
    candidates.append({
        "trading_pair": combo["trading_pair"],
        "count": combo["count"],
        "first_ts": combo["first_ts"],
        "last_ts": combo["last_ts"],
        "data_days": data_days,
    })

print(f"\n{'='*60}")
print(f"Found {len(candidates)} pairs with >= {MIN_DATA_DAYS} days of {INTERVAL} data on {CONNECTOR}:")
print(f"{'='*60}")
for c in candidates:
    print(f"  {c['trading_pair']:15s}  {c['count']:>8,} candles  {c['data_days']:5.1f} days")
print(f"\nTotal pairs to optimize: {len(candidates)}")


Found 13 pairs with >= 28 days of 5m data on nonkyc:
  ARRR-USDT          52,149 candles  181.1 days
  BNB-USDT           52,111 candles  180.9 days
  BTC-USDT           55,904 candles  194.1 days
  DASH-USDT          52,140 candles  181.0 days
  DOGE-USDT          55,774 candles  193.7 days
  ETH-USDT           55,817 candles  193.8 days
  LINK-USDT          52,423 candles  182.0 days
  LTC-USDT           52,559 candles  182.5 days
  SAL-USDT           55,706 candles  193.4 days
  SOL-USDT           55,785 candles  193.7 days
  UNI-USDT           52,424 candles  182.0 days
  XMR-USDT           55,615 candles  193.1 days
  XRP-USDT           52,414 candles  182.0 days

Total pairs to optimize: 13


## 3. Sweep: Optimize Each Pair

For each pair, the sweep:
1. Loads and validates candles
2. Auto-scales walk-forward windows to fit available data
3. Runs 3000 Optuna trials (walk-forward, stress OFF)
4. Stress-tests top 50 candidates
5. Records the best stress-validated result

In [ ]:
from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.optuna.study import create_study
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.callbacks import DegeneracyCheckCallback, TrialLoggingCallback
from pmm_lab.optuna.canonicalizer import canonicalize_params
from pmm_lab.objective.stress import run_stress_tests
from pmm_lab.objective.walkforward import run_walk_forward
from pmm_lab.objective.objective import REJECT_SCORE
from pmm_lab.export.hb_yaml import export_yaml, ExportParams
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.metrics.metrics import compute_metrics
from pmm_lab.objective.objective import objective_v1
from pmm_lab.report.report_md import generate_report, run_stop_ship_checks
from pmm_lab.sim.runner import CandleSimRunner
from tqdm.notebook import tqdm

from optuna.storages import RDBStorage

if OPTUNA_STORAGE:
    optuna_storage = RDBStorage(
        url=OPTUNA_STORAGE,
        engine_kwargs={"pool_size": 32, "max_overflow": 16},
    )
else:
    optuna_storage = None

rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

for pair_idx, pair_info in enumerate(tqdm(candidates, desc="Pairs", unit="pair")):
    pair = pair_info["trading_pair"]
    print(f"\n{'═'*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {CONNECTOR} / {pair} / {INTERVAL}")
    print(f"{'═'*60}")

    pair_start = time.time()

    # ── Load candles ──
    try:
        query = DataQuery(connector=CONNECTOR, trading_pair=pair, interval=INTERVAL)
        candles = loader.load_range(query)
        audit = validate_candles(candles, interval=INTERVAL, strict=True)
        if not audit.passed_strict:
            print(f"  SKIP: audit failed — {audit.failure_reasons}")
            sweep_results.append({"pair": pair, "status": "audit_fail", "robust_score": None})
            continue
        dataset_hash = hash_candles(candles)
    except Exception as e:
        print(f"  SKIP: load failed — {e}")
        sweep_results.append({"pair": pair, "status": "load_fail", "robust_score": None})
        continue

    # ── Exchange rules ──
    try:
        pair_rules = resolve_pair_rules(rules_db, CONNECTOR, pair)
    except KeyError:
        try:
            pair_rules = resolve_pair_rules(rules_db, CONNECTOR, "DEFAULT")
        except KeyError:
            print(f"  SKIP: no exchange rules for {CONNECTOR}/{pair}")
            sweep_results.append({"pair": pair, "status": "no_rules", "robust_score": None})
            continue

    ref_price = float(np.median(candles["close"]))

    # ── Auto-scale walk-forward windows ──
    dataset_days = len(candles) * BAR_INTERVAL_SECONDS / 86400
    if dataset_days >= 120:
        train_days, test_days, step_days = 42.0, 14.0, 14.0
    elif dataset_days >= 60:
        train_days, test_days, step_days = 21.0, 7.0, 7.0
    elif dataset_days >= 28:
        train_days, test_days, step_days = 10.0, 4.0, 4.0
    else:
        print(f"  SKIP: only {dataset_days:.1f} days of data")
        sweep_results.append({"pair": pair, "status": "insufficient_data", "robust_score": None})
        continue

    print(f"  Candles: {len(candles):,}  Days: {dataset_days:.1f}  "
          f"WF: {train_days}/{test_days}/{step_days}d  Ref: {ref_price:,.4f}")

    # ── Phase 1: Optimization ──
    study_name = f"{CONNECTOR}_{pair}_{INTERVAL}_sweep_nonkyc_v2"

    try:
        study = create_study(
            study_name=study_name,
            storage_url=optuna_storage if optuna_storage else None,
            n_startup_trials=int(N_TRIALS * 0.10),
        )

        if WORKER_MODEL == "processes" and N_WORKERS > 1 and OPTUNA_STORAGE:
            from pmm_lab.optuna.parallel import run_parallel_optimization

            worker_results = run_parallel_optimization(
                study_name=study_name,
                storage_url=OPTUNA_STORAGE,
                n_total_trials=N_TRIALS,
                n_workers=N_WORKERS,
                objective_factory=create_objective,
                factory_kwargs=dict(
                    candles=candles,
                    pair_rules=pair_rules,
                    bar_interval_seconds=BAR_INTERVAL_SECONDS,
                    dataset_hash=dataset_hash,
                    reference_price=ref_price,
                    train_days=train_days,
                    test_days=test_days,
                    step_days=step_days,
                    run_stress=False,
                    controller_compat=PHASE1_CONTROLLER_COMPAT,
                ),
                sampler_seed=12345,
                n_startup_trials=int(N_TRIALS * 0.10),
            )

            # Reload the study to get all trials
            study = optuna.load_study(
                study_name=study_name,
                storage=optuna_storage,
            )

            for wr in worker_results:
                if wr.error:
                    print(f"  Worker {wr.worker_id} ERROR: {wr.error}")
                else:
                    print(f"  Worker {wr.worker_id}: {wr.wall_time:.1f}s")

        else:
            # Fallback: single-process, single-thread
            objective_fn = create_objective(
                candles=candles,
                pair_rules=pair_rules,
                bar_interval_seconds=BAR_INTERVAL_SECONDS,
                dataset_hash=dataset_hash,
                reference_price=ref_price,
                train_days=train_days,
                test_days=test_days,
                step_days=step_days,
                run_stress=False,
                controller_compat=PHASE1_CONTROLLER_COMPAT,
            )

            study.optimize(
                objective_fn,
                n_trials=N_TRIALS,
                callbacks=[DegeneracyCheckCallback()],
                catch=(Exception,),
                n_jobs=1,  # ALWAYS 1 — parallelism comes from processes, not threads
                show_progress_bar=True,
            )

        n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        best_val = study.best_value if study.best_trial else REJECT_SCORE
        print(f"  Phase 1: {n_complete} complete, {n_pruned} pruned, best={best_val:.4f}")
    except Exception as e:
        print(f"  SKIP: optimization failed — {e}")
        sweep_results.append({"pair": pair, "status": "optim_fail", "robust_score": None})
        continue

    # ── Phase 2: Stress top N ──
    try:
        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        ranked = sorted(completed, key=lambda t: t.value, reverse=True)
        top_trials = ranked[:min(TOP_N, len(ranked))]

        top_candidates = []
        for trial in top_trials:
            config, reject = canonicalize_params(trial.params, pair_rules, ref_price)
            if config is not None:
                top_candidates.append({
                    "trial_number": trial.number,
                    "phase1_score": trial.value,
                    "params": trial.params,
                    "config": config,
                })

        if not top_candidates:
            print(f"  SKIP: no valid configs to stress test")
            sweep_results.append({"pair": pair, "status": "no_valid_configs", "robust_score": None})
            continue

        print(f"  Stress testing {len(top_candidates)} candidates...")
        for candidate in tqdm(top_candidates, desc=f"  Stress {pair}", unit="cfg", leave=False):
            stress_report = run_stress_tests(
                candles=candles,
                config=candidate["config"],
                pair_rules=pair_rules,
                bar_interval_seconds=BAR_INTERVAL_SECONDS,
            )
            candidate["stress_report"] = stress_report
            candidate["baseline_score"] = stress_report.baseline_objective.raw_score
            candidate["worst_scenario"] = stress_report.worst_scenario
            candidate["worst_score"] = stress_report.worst_score
            candidate["robust_score"] = (
                0.5 * stress_report.baseline_objective.raw_score
                + 0.5 * stress_report.worst_score
            )

        # Find best by robust score
        top_candidates.sort(key=lambda c: c["robust_score"], reverse=True)
        best = top_candidates[0]
        best_config = best["config"]
        best_stress = best["stress_report"]
        bm = best_stress.baseline_metrics

        pair_elapsed = time.time() - pair_start
        print(f"  Best: trial {best['trial_number']}  robust={best['robust_score']:.4f}  "
              f"PnL={bm.pnl_pct:.2f}%  trades={bm.trade_count}  ({pair_elapsed/60:.1f}min)")

    except Exception as e:
        print(f"  SKIP: stress testing failed — {e}")
        sweep_results.append({"pair": pair, "status": "stress_fail", "robust_score": None})
        continue

    # ── Record result ──
    result_entry = {
        "pair": pair,
        "status": "complete",
        "robust_score": best["robust_score"],
        "baseline_score": best["baseline_score"],
        "worst_score": best["worst_score"],
        "worst_scenario": best["worst_scenario"],
        "pnl_pct": bm.pnl_pct,
        "sharpe": bm.sharpe,
        "max_dd_pct": bm.max_drawdown_pct,
        "trade_count": bm.trade_count,
        "total_fees": bm.total_fees_quote,
        "profit_factor": bm.profit_factor,
        "trial_number": best["trial_number"],
        "best_config": best_config,
        "best_params": best["params"],
        "best_stress": best_stress,
        "dataset_hash": dataset_hash,
        "n_candles": len(candles),
        "dataset_days": dataset_days,
        "train_days": train_days,
        "test_days": test_days,
        "step_days": step_days,
        "study_name": study_name,
    }
    sweep_results.append(result_entry)

    # ── Export if profitable ──
    if best["robust_score"] >= MIN_ROBUST_SCORE:
        try:
            export_params = ExportParams(
                connector_name=CONNECTOR,
                trading_pair=pair,
                candles_connector=CONNECTOR,
                candles_trading_pair=pair,
                interval=INTERVAL,
            )

            yaml_path = export_yaml(
                config=best_config,
                output_path=f"artifacts/sweep/{CONNECTOR}/{pair}_{INTERVAL}_best.yml",
                export_params=export_params,
                metadata={
                    "dataset_hash": dataset_hash,
                    "trial": best["trial_number"],
                    "phase1_score": best["phase1_score"],
                    "robust_score": best["robust_score"],
                    "worst_scenario": best["worst_scenario"],
                    "worst_score": best["worst_score"],
                    "sweep_date": datetime.utcnow().isoformat(),
                },
            )

            # Walk-forward for report
            wf_result = run_walk_forward(
                candles=candles, config=best_config, pair_rules=pair_rules,
                bar_interval_seconds=BAR_INTERVAL_SECONDS, dataset_hash=dataset_hash,
                train_days=train_days, test_days=test_days, step_days=step_days,
            )

            # Metrics for report
            runner = CandleSimRunner(best_config, pair_rules)
            sim_result = runner.run(candles)
            best_metrics = compute_metrics(sim_result, best_config.total_amount_quote, candles, BAR_INTERVAL_SECONDS)
            best_obj = objective_v1(best_metrics)

            checks = run_stop_ship_checks(
                best_metrics=best_metrics, best_objective=best_obj,
                walkforward_result=wf_result, stress_report=best_stress,
                dataset_audit=audit,
            )

            generate_report(
                study_name=study_name,
                dataset_summary={
                    "connector": CONNECTOR, "trading_pair": pair, "interval": INTERVAL,
                    "n_candles": len(candles), "dataset_hash": dataset_hash,
                    "n_trials_phase1": N_TRIALS, "n_candidates_stressed": len(top_candidates),
                },
                best_params=best["params"], best_metrics=best_metrics, best_objective=best_obj,
                walkforward_result=wf_result, stress_report=best_stress,
                stop_ship_checks=checks,
                output_path=f"artifacts/sweep/{CONNECTOR}/{pair}_{INTERVAL}_report.md",
            )

            result_entry["exported"] = True
            result_entry["yaml_path"] = yaml_path
            all_pass = all(checks.values())
            result_entry["all_checks_pass"] = all_pass
            print(f"  EXPORTED  yaml={yaml_path}  checks={'ALL PASS' if all_pass else 'SOME FAIL'}")
        except Exception as e:
            print(f"  Export failed: {e}")
            result_entry["exported"] = False
    else:
        result_entry["exported"] = False
        print(f"  NOT PROFITABLE (robust={best['robust_score']:.4f} < {MIN_ROBUST_SCORE})")

total_elapsed = time.time() - sweep_start
print(f"\n{'═'*60}")
print(f"SWEEP COMPLETE: {len(candidates)} pairs in {total_elapsed/60:.1f} minutes")
print(f"{'═'*60}")

Pairs:   0%|          | 0/13 [00:00<?, ?pair/s]


════════════════════════════════════════════════════════════
  [1/13] nonkyc / ARRR-USDT / 5m
════════════════════════════════════════════════════════════
  Candles: 52,149  Days: 181.1  WF: 42.0/14.0/14.0d  Ref: 0.2857


Bar 18132: no orders placed (this warning will not repeat)


  Worker 1: 611.1s
  Worker 3: 612.6s
  Worker 0: 635.5s
  Worker 2: 636.2s
  Phase 1: 2702 complete, 298 pruned, best=383.1030
  Stress testing 50 candidates...


  Stress ARRR-USDT:   0%|          | 0/50 [00:00<?, ?cfg/s]

## 4. Results Summary

In [ ]:
# Build summary table
summary_rows = []
for r in sweep_results:
    row = {
        "Pair": r["pair"],
        "Status": r["status"],
    }
    if r["status"] == "complete":
        row.update({
            "Robust": f"{r['robust_score']:.2f}",
            "PnL%": f"{r['pnl_pct']:.2f}",
            "Sharpe": f"{r['sharpe']:.2f}",
            "MaxDD%": f"{r['max_dd_pct']:.2f}",
            "Trades": r["trade_count"],
            "PF": f"{r['profit_factor']:.2f}" if r["profit_factor"] != float('inf') else "\u221e",
            "Fees": f"{r['total_fees']:.2f}",
            "WorstStress": r.get("worst_scenario", ""),
            "Exported": "\u2713" if r.get("exported") else "\u2717",
            "Checks": "PASS" if r.get("all_checks_pass") else "\u2014",
        })
    else:
        row.update({k: "\u2014" for k in ["Robust", "PnL%", "Sharpe", "MaxDD%", "Trades",
                                       "PF", "Fees", "WorstStress", "Exported", "Checks"]})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

# Sort: completed + exported first, then by robust score
def sort_key(row):
    if row["Status"] != "complete":
        return (2, 0)
    if row["Exported"] == "\u2713":
        return (0, -float(row["Robust"]))
    return (1, -float(row["Robust"]))

summary_df["_sort"] = summary_df.apply(sort_key, axis=1)
summary_df = summary_df.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

print(f"{'='*60}")
print(f"  SWEEP RESULTS: {CONNECTOR} / {QUOTE_ASSET} / {INTERVAL}")
print(f"{'='*60}\n")

n_complete = len([r for r in sweep_results if r["status"] == "complete"])
n_exported = len([r for r in sweep_results if r.get("exported")])
n_profitable = len([r for r in sweep_results if r["status"] == "complete" and r["robust_score"] >= MIN_ROBUST_SCORE])

print(f"  Total pairs scanned : {len(candidates)}")
print(f"  Completed           : {n_complete}")
print(f"  Profitable          : {n_profitable} (robust score >= {MIN_ROBUST_SCORE})")
print(f"  Exported            : {n_exported}")
print()

display(summary_df)

## 5. Profitable Pairs Detail

In [ ]:
profitable = [r for r in sweep_results if r["status"] == "complete"
              and r["robust_score"] is not None and r["robust_score"] >= MIN_ROBUST_SCORE]
profitable.sort(key=lambda r: r["robust_score"], reverse=True)

if not profitable:
    print("No profitable pairs found in this sweep.")
    print(f"Try adjusting MIN_ROBUST_SCORE (currently {MIN_ROBUST_SCORE}) or running on a different connector/interval.")
else:
    for i, r in enumerate(profitable):
        print(f"\n{'\u2500'*60}")
        print(f"  #{i+1}  {r['pair']}  (robust={r['robust_score']:.4f})")
        print(f"{'\u2500'*60}")
        print(f"  PnL %       : {r['pnl_pct']:.4f}")
        print(f"  Sharpe      : {r['sharpe']:.4f}")
        print(f"  Max DD %    : {r['max_dd_pct']:.4f}")
        print(f"  Trades      : {r['trade_count']}")
        print(f"  Profit Fac. : {r['profit_factor']:.4f}")
        print(f"  Fees        : {r['total_fees']:.4f}")
        print(f"  Worst stress: {r['worst_scenario']} ({r['worst_score']:.4f})")
        print(f"  Data        : {r['n_candles']:,} candles, {r['dataset_days']:.1f} days")
        if r.get("yaml_path"):
            print(f"  YAML        : {r['yaml_path']}")
        print(f"  Checks      : {'ALL PASS' if r.get('all_checks_pass') else 'SOME FAILED'}")

    print(f"\n{'='*60}")
    print(f"  {len(profitable)} profitable pair(s) found!")
    print(f"  Check artifacts/sweep/{CONNECTOR}/ for configs and reports.")
    print(f"{'='*60}")

## 6. Next Steps

For each exported pair:
1. **Review the report** in `artifacts/sweep/<connector>/`
2. **Verify stop-ship checks** all pass
3. **Paper trade** using Hummingbot's paper trading mode
4. **Monitor** for at least 1 week before live trading
5. **Compare** live performance to backtest expectations

To re-run for a different connector, change `CONNECTOR` in the configuration cell and Run All.